# Cardiac AI V4 - Danh gia tren tap Test & Tim nguong toi uu
**Chay doc lap, khong can train lai**

## Cell 1 - Cai thu vien

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb'])
print('Done')

## Cell 2 - Imports & Config

In [ ]:
import os, json, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

import wfdb
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import (f1_score, roc_auc_score, average_precision_score,
                              confusion_matrix, classification_report,
                              precision_recall_curve)
from sklearn.model_selection import train_test_split
from collections import defaultdict

# Config (phai khop voi luc train V4)
SEED         = 42
NUM_LEADS    = 12
SEQ_LEN      = 4096
DROPOUT      = 0.15
MS_CHANNELS  = [128, 256, 512, 512]
TRANS_DIM    = 384
TRANS_HEADS  = 8
TRANS_LAYERS = 8
TRANS_FF_DIM = 1536
NUM_CLASSES  = 27
BATCH_SIZE   = 32
NUM_WORKERS  = 4

DATA_DIR    = ('/kaggle/input/datasets/gamalasran/physionet-challenge-2020/'
               'classification-of-12-lead-ecgs-the-physionetcomputing-in-'
               'cardiology-challenge-2020-1.0.2/training')
MAPPING_CSV = ('/kaggle/input/datasets/bjoernjostein/physionet-snomed-mappings/'
               'SNOMED_mappings_scored.csv')
MODEL_PATH     = '/kaggle/input/datasets/tuyenngoc12/cardiac-ai-v4-model/cardiac_v4_best.pth'
OUTPUT_DIR  = '/kaggle/working/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('Config OK')

## Cell 3 - SNOMED & Scan files

In [ ]:
df_map = pd.read_csv(MAPPING_CSV, sep=';')
SNOMED_TO_IDX = {}
IDX_TO_NAME   = {}
IDX_TO_VIET   = {
    0:'Bloc nhi that do 1', 1:'Rung nhi', 2:'Cuong nhi',
    3:'Nhip cham', 4:'Bloc nhanh phai hoan toan',
    5:'Bloc nhanh phai khong hoan toan', 6:'Bloc phan nhanh trai truoc',
    7:'Lech truc trai', 8:'Bloc nhanh trai', 9:'Dien the QRS thap',
    10:'Roi loan dan truyen trong that', 11:'Khoang PR',
    12:'Ngoai tam thu nhi', 13:'Ngoai tam thu that',
    14:'Khoang PR keo dai', 15:'Khoang QT keo dai',
    16:'Song Q bat thuong', 17:'Lech truc phai',
    18:'Bloc nhanh phai', 19:'Roi loan nhip xoang',
    20:'Nhip cham xoang', 21:'Nhip xoang binh thuong',
    22:'Nhip nhanh xoang', 23:'Nhip som tren that',
    24:'Song T bat thuong', 25:'Song T dao nguoc',
    26:'Nhip som that'
}

for idx, row in df_map.iterrows():
    try:
        code_val = int(row['SNOMED CT Code'])
        SNOMED_TO_IDX[code_val] = idx
        IDX_TO_NAME[idx] = str(row['Abbreviation'])
    except: pass

def parse_hea_labels(hea_path):
    labels = []
    try:
        with open(hea_path, 'r', errors='ignore') as fh:
            for line in fh:
                if line.startswith('#') and 'Dx:' in line:
                    for tok in line.split('Dx:')[-1].split(','):
                        tok = tok.strip()
                        if tok.isdigit() and int(tok) in SNOMED_TO_IDX:
                            labels.append(SNOMED_TO_IDX[int(tok)])
    except: pass
    return list(set(labels))

print('Scanning files...')
records = []
for root, _, files in os.walk(DATA_DIR):
    for fname in files:
        if not fname.endswith('.hea'): continue
        hea = os.path.join(root, fname)
        mat = hea.replace('.hea', '.mat')
        if not os.path.exists(mat): continue
        lbls = parse_hea_labels(hea)
        if lbls:
            records.append({'hea_path':hea, 'mat_path':mat, 'labels':lbls})

df_records   = pd.DataFrame(records)
label_matrix = np.zeros((len(df_records), NUM_CLASSES), dtype=np.float32)
for i, row in df_records.iterrows():
    for lbl in row['labels']:
        if lbl < NUM_CLASSES: label_matrix[i, lbl] = 1.0

# Tao lai dung split SEED=42 (khop voi luc train)
idx_all = np.arange(len(df_records))
train_idx, tmp_idx = train_test_split(idx_all, test_size=0.20, random_state=SEED)
val_idx,  test_idx = train_test_split(tmp_idx,  test_size=0.50, random_state=SEED)

print(f'Records: {len(df_records):,}')
print(f'Test set: {len(test_idx):,} samples')

## Cell 4 - Dataset & Model

In [ ]:
class ECGDataset(Dataset):
    def __init__(self, df, label_matrix, seq_len=SEQ_LEN):
        self.df       = df.reset_index(drop=True)
        self.orig_idx = df.index.to_numpy()
        self.lm       = label_matrix
        self.seq_len  = seq_len

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        mat_path = self.df.iloc[idx]['mat_path']
        try:
            rec = wfdb.rdrecord(mat_path.replace('.mat',''))
            sig = rec.p_signal.astype(np.float32)
        except:
            sig = np.zeros((self.seq_len, NUM_LEADS), dtype=np.float32)
        mu  = sig.mean(0, keepdims=True)
        std = sig.std(0,  keepdims=True) + 1e-8
        sig = (sig - mu) / std
        T   = sig.shape[0]
        if T >= self.seq_len:
            start = (T - self.seq_len) // 2
            sig = sig[start:start+self.seq_len]
        else:
            sig = np.vstack([sig, np.zeros((self.seq_len-T, NUM_LEADS), np.float32)])
        x   = torch.from_numpy(sig.T)
        lbl = torch.from_numpy(self.lm[self.orig_idx[idx]].copy())
        return x, lbl

# Model V4
class MultiScaleConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=2):
        super().__init__()
        b = out_ch // 4
        self.b1 = nn.Sequential(nn.Conv1d(in_ch,b,3,padding=1,bias=False),
                                 nn.GroupNorm(min(8,b),b),nn.GELU())
        self.b2 = nn.Sequential(nn.Conv1d(in_ch,b,9,padding=4,bias=False),
                                 nn.GroupNorm(min(8,b),b),nn.GELU())
        self.b3 = nn.Sequential(nn.Conv1d(in_ch,b,19,padding=9,bias=False),
                                 nn.GroupNorm(min(8,b),b),nn.GELU())
        self.b4 = nn.Sequential(nn.AvgPool1d(3,stride=1,padding=1),
                                 nn.Conv1d(in_ch,b,1,bias=False),
                                 nn.GroupNorm(min(8,b),b),nn.GELU())
        self.fusion = nn.Sequential(nn.Conv1d(out_ch,out_ch,3,padding=1,bias=False),
                                     nn.GroupNorm(min(8,out_ch),out_ch),nn.GELU())
        self.pool = nn.MaxPool1d(pool)
        self.shortcut = (nn.Sequential(nn.Conv1d(in_ch,out_ch,1,bias=False),
                                        nn.GroupNorm(min(8,out_ch),out_ch))
                         if in_ch!=out_ch else nn.Identity())
    def forward(self,x):
        return self.pool(self.fusion(torch.cat([self.b1(x),self.b2(x),
                                                self.b3(x),self.b4(x)],1))
                         + self.shortcut(x))

class PositionalEncoding(nn.Module):
    def __init__(self,d_model,max_len=512,dropout=DROPOUT):
        super().__init__()
        self.dropout=nn.Dropout(dropout)
        pe=torch.zeros(max_len,d_model)
        pos=torch.arange(max_len).unsqueeze(1).float()
        div=torch.exp(torch.arange(0,d_model,2).float()*(-np.log(10000.0)/d_model))
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer('pe',pe.unsqueeze(0))
    def forward(self,x): return self.dropout(x+self.pe[:,:x.size(1)])

class ECGModelV4(nn.Module):
    def __init__(self,num_classes=NUM_CLASSES):
        super().__init__()
        ch=[NUM_LEADS]+MS_CHANNELS
        self.cnn=nn.Sequential(*[MultiScaleConvBlock(ch[i],ch[i+1])
                                  for i in range(len(MS_CHANNELS))])
        self.proj=nn.Sequential(nn.Linear(MS_CHANNELS[-1],TRANS_DIM),
                                  nn.LayerNorm(TRANS_DIM))
        self.pos_enc=PositionalEncoding(TRANS_DIM,max_len=1024)
        enc=nn.TransformerEncoderLayer(d_model=TRANS_DIM,nhead=TRANS_HEADS,
            dim_feedforward=TRANS_FF_DIM,dropout=DROPOUT,
            activation='gelu',batch_first=True,norm_first=True)
        self.transformer=nn.TransformerEncoder(enc,num_layers=TRANS_LAYERS)
        self.head=nn.Sequential(nn.LayerNorm(TRANS_DIM),nn.Dropout(DROPOUT),
                                  nn.Linear(TRANS_DIM,512),nn.GELU(),
                                  nn.Dropout(DROPOUT/2),nn.Linear(512,num_classes))
    def forward(self,x):
        x=self.cnn(x); x=x.permute(0,2,1)
        x=self.proj(x); x=self.pos_enc(x)
        x=self.transformer(x); x=x.mean(1)
        return self.head(x)

model = ECGModelV4().to(device)
ckpt  = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(ckpt['state'])
model.eval()
print(f'Model V4 loaded! epoch={ckpt["epoch"]} val_F1={ckpt["score"]:.4f}')
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

## Cell 5 - Predict tren tap Test (TTA 5x)

In [ ]:
test_ds = ECGDataset(df_records.iloc[test_idx], label_matrix)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

def predict_loader(loader, augment=False):
    all_probs, all_targets = [], []
    with torch.no_grad():
        for x, labels in tqdm(loader, leave=False):
            if augment:
                # Augment nhe cho TTA
                x = x + torch.randn_like(x) * 0.03
                x = x * (0.9 + torch.rand(x.shape[0],1,1) * 0.2)
            logits = model(x.to(device))
            all_probs.append(torch.sigmoid(logits).float().cpu().numpy())
            all_targets.append(labels.numpy())
    return np.concatenate(all_probs), np.concatenate(all_targets)

print('Predicting tren test set (TTA 5x)...')
y_prob, y_true = predict_loader(test_loader, augment=False)

# TTA rounds
N_TTA = 5
for t in range(N_TTA):
    p_aug, _ = predict_loader(test_loader, augment=True)
    y_prob += p_aug
    print(f'TTA round {t+1}/{N_TTA} done')

y_prob /= (N_TTA + 1)
print(f'\nTest set: {y_true.shape[0]} samples')
print(f'Positive labels: {y_true.sum(0).astype(int).tolist()}')

## Cell 6 - Tim nguong toi uu tung class

In [ ]:
print('Tim nguong toi uu tung class...')

results = []
best_thresholds = []

for ci in range(NUM_CLASSES):
    n_pos = int(y_true[:, ci].sum())
    if n_pos == 0:
        best_thresholds.append(0.5)
        results.append({'class': ci, 'name': IDX_TO_NAME.get(ci,'?'),
                        'viet': IDX_TO_VIET.get(ci,'?'),
                        'f1': 0.0, 'precision': 0.0, 'recall': 0.0,
                        'auc': float('nan'), 'threshold': 0.5, 'n_test': 0})
        continue

    # Tim threshold toi uu bang Precision-Recall curve
    precision_arr, recall_arr, thresh_arr = precision_recall_curve(
        y_true[:, ci], y_prob[:, ci])

    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.05, 0.95, 0.025):
        pred = (y_prob[:, ci] >= t).astype(int)
        f1   = f1_score(y_true[:, ci], pred, zero_division=0)
        if f1 > best_f1: best_f1, best_t = f1, float(t)

    best_thresholds.append(best_t)
    pred_best = (y_prob[:, ci] >= best_t).astype(int)
    prec = precision_arr[np.argmin(np.abs(thresh_arr - best_t))] if len(thresh_arr) > 0 else 0
    rec  = recall_arr[np.argmin(np.abs(thresh_arr - best_t))]    if len(thresh_arr) > 0 else 0

    try:    auc = roc_auc_score(y_true[:, ci], y_prob[:, ci])
    except: auc = float('nan')

    results.append({'class': ci, 'name': IDX_TO_NAME.get(ci,'?'),
                    'viet': IDX_TO_VIET.get(ci,'?'),
                    'f1': round(best_f1,4), 'precision': round(float(prec),4),
                    'recall': round(float(rec),4), 'auc': round(auc,4),
                    'threshold': best_t, 'n_test': n_pos})

df_results = pd.DataFrame(results).sort_values('f1', ascending=False)

# Tinh tong quat
y_pred_tuned = (y_prob >= np.array(best_thresholds)).astype(int)
macro_f1 = f1_score(y_true, y_pred_tuned, average='macro', zero_division=0)
micro_f1 = f1_score(y_true, y_pred_tuned, average='micro', zero_division=0)
mask     = y_true.sum(0) > 0
try:    macro_auc = roc_auc_score(y_true[:,mask], y_prob[:,mask], average='macro')
except: macro_auc = 0.0
try:    mAP = average_precision_score(y_true[:,mask], y_prob[:,mask], average='macro')
except: mAP = 0.0

print(f'\n{"="*60}')
print(f'TONG QUAN TAP TEST (TTA 5x + Threshold Tuning)')
print(f'{"="*60}')
print(f'Macro F1 : {macro_f1:.4f}')
print(f'Micro F1 : {micro_f1:.4f}')
print(f'Macro AUC: {macro_auc:.4f}')
print(f'mAP      : {mAP:.4f}')
print(f'{"="*60}')
print(f'\nKet qua tung benh:')
print(df_results[['name','viet','f1','auc','threshold','n_test']].to_string(index=False))

## Cell 7 - Ve bieu do danh gia

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 8), facecolor='#0f172a')

# 1. Bar chart F1 tung class
ax = axes[0]
ax.set_facecolor('#0f172a')
df_plot = df_results.sort_values('f1', ascending=True)
colors  = ['#22c55e' if f>=0.7 else '#f59e0b' if f>=0.5 else '#ef4444'
            for f in df_plot.f1]
bars = ax.barh(range(len(df_plot)), df_plot.f1, color=colors, height=0.7)
ax.set_yticks(range(len(df_plot)))
ax.set_yticklabels(df_plot.name, color='#94a3b8', fontsize=8)
ax.set_xlabel('F1 Score', color='#94a3b8')
ax.set_title('F1 Score tung benh', color='white', fontweight='bold')
ax.set_xlim(0, 1.1)
ax.axvline(x=macro_f1, color='white', linestyle='--', alpha=0.7, linewidth=1.5)
ax.text(macro_f1+0.01, len(df_plot)-1, f'Mean={macro_f1:.3f}',
        color='white', fontsize=8)
for sp in ax.spines.values(): sp.set_color('#1e293b')
ax.tick_params(colors='#475569')
ax.grid(axis='x', alpha=0.15, color='#334155')
legend_patches = [
    mpatches.Patch(color='#22c55e', label='Tot (>=0.7)'),
    mpatches.Patch(color='#f59e0b', label='Trung binh (0.5-0.7)'),
    mpatches.Patch(color='#ef4444', label='Can cai thien (<0.5)')]
ax.legend(handles=legend_patches, facecolor='#1e293b',
           labelcolor='white', fontsize=7, loc='lower right')

# 2. AUC tung class
ax = axes[1]
ax.set_facecolor('#0f172a')
df_auc = df_results.dropna(subset=['auc']).sort_values('auc', ascending=True)
colors2 = ['#22c55e' if a>=0.9 else '#f59e0b' if a>=0.8 else '#ef4444'
            for a in df_auc.auc]
ax.barh(range(len(df_auc)), df_auc.auc, color=colors2, height=0.7)
ax.set_yticks(range(len(df_auc)))
ax.set_yticklabels(df_auc.name, color='#94a3b8', fontsize=8)
ax.set_xlabel('AUC', color='#94a3b8')
ax.set_title('ROC-AUC tung benh', color='white', fontweight='bold')
ax.set_xlim(0.5, 1.05)
ax.axvline(x=macro_auc, color='white', linestyle='--', alpha=0.7, linewidth=1.5)
ax.text(macro_auc+0.003, len(df_auc)-1, f'Mean={macro_auc:.3f}',
        color='white', fontsize=8)
for sp in ax.spines.values(): sp.set_color('#1e293b')
ax.tick_params(colors='#475569')
ax.grid(axis='x', alpha=0.15, color='#334155')

# 3. Nguong tung class
ax = axes[2]
ax.set_facecolor('#0f172a')
df_thr = df_results.sort_values('threshold', ascending=True)
ax.barh(range(len(df_thr)), df_thr.threshold, color='#3b82f6', height=0.7, alpha=0.8)
ax.set_yticks(range(len(df_thr)))
ax.set_yticklabels(df_thr.name, color='#94a3b8', fontsize=8)
ax.set_xlabel('Threshold', color='#94a3b8')
ax.set_title('Nguong toi uu tung benh', color='white', fontweight='bold')
ax.set_xlim(0, 1.0)
ax.axvline(x=0.5, color='white', linestyle='--', alpha=0.5, linewidth=1)
for sp in ax.spines.values(): sp.set_color('#1e293b')
ax.tick_params(colors='#475569')
ax.grid(axis='x', alpha=0.15, color='#334155')

plt.suptitle(f'Cardiac AI V4 - Danh gia tap Test | Macro F1={macro_f1:.4f} | AUC={macro_auc:.4f}',
             color='white', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/evaluation_v4.png', dpi=120,
            bbox_inches='tight', facecolor='#0f172a')
plt.show()
plt.close()
print('Da luu bieu do: evaluation_v4.png')

## Cell 8 - Luu ket qua & Thresholds moi

In [ ]:
# Luu thresholds moi (chinh xac hon voi step 0.025)
new_thresholds = {str(i): float(t) for i,t in enumerate(best_thresholds)}
with open(f'{OUTPUT_DIR}/best_thresholds_v4_refined.json', 'w') as f:
    json.dump(new_thresholds, f, indent=2)

# Luu per-class report
df_results.to_csv(f'{OUTPUT_DIR}/per_class_v4_refined.csv', index=False)

# In tom tat
print('='*65)
print('TOM TAT KET QUA')
print('='*65)
print(f'Model         : V4 - Multi-scale CNN + Transformer (20M)')
print(f'Test samples  : {y_true.shape[0]:,}')
print(f'TTA rounds    : 5x')
print(f'Threshold step: 0.025 (chinh xac hon 0.05 cu)')
print(f'')
print(f'Macro F1  : {macro_f1:.4f}')
print(f'Micro F1  : {micro_f1:.4f}')
print(f'Macro AUC : {macro_auc:.4f}')
print(f'mAP       : {mAP:.4f}')
print(f'')
print(f'F1 >= 0.7 : {(df_results.f1>=0.7).sum()} classes')
print(f'F1 >= 0.5 : {(df_results.f1>=0.5).sum()} classes')
print(f'F1 < 0.3  : {(df_results.f1<0.3).sum()} classes')
print(f'')
print(f'Saved:')
print(f'  {OUTPUT_DIR}/best_thresholds_v4_refined.json')
print(f'  {OUTPUT_DIR}/per_class_v4_refined.csv')
print(f'  {OUTPUT_DIR}/evaluation_v4.png')
print('='*65)